# Advanced Schema Generation and Financial Use Cases

In this notebook, we'll explore more complex financial schemas and advanced constraint techniques using Outlines.

## Setup

In [ ]:
import outlines
import json
from pydantic import BaseModel, Field, validator
from typing import List, Literal, Optional, Union
from datetime import datetime, date
from enum import Enum
import time
import random

In [ ]:
# Initialize model
model = outlines.models.transformers("microsoft/DialoGPT-medium")
print("Model loaded successfully!")

## 1. Complex Financial Risk Assessment Schema

Let's create a comprehensive risk assessment schema with multiple nested components:

In [ ]:
# Define enums for better type safety
class RiskCategory(str, Enum):
    VERY_LOW = "very_low"
    LOW = "low" 
    MEDIUM = "medium"
    HIGH = "high"
    VERY_HIGH = "very_high"

class IndustryClassification(str, Enum):
    TECHNOLOGY = "technology"
    HEALTHCARE = "healthcare"
    FINANCE = "finance"
    ENERGY = "energy"
    CONSUMER_GOODS = "consumer_goods"
    REAL_ESTATE = "real_estate"
    UTILITIES = "utilities"

class CreditRating(str, Enum):
    AAA = "AAA"
    AA = "AA" 
    A = "A"
    BBB = "BBB"
    BB = "BB"
    B = "B"
    CCC = "CCC"
    D = "D"

# Define nested models
class FinancialMetrics(BaseModel):
    revenue: float = Field(ge=0, description="Annual revenue in millions USD")
    ebitda: float = Field(description="EBITDA in millions USD")
    debt_to_equity: float = Field(ge=0, description="Debt to equity ratio")
    current_ratio: float = Field(ge=0, description="Current ratio")
    return_on_equity: float = Field(ge=-100, le=100, description="ROE percentage")

class RiskFactor(BaseModel):
    factor_name: str = Field(max_length=100)
    impact_score: float = Field(ge=0, le=10, description="Impact score from 0-10")
    probability: float = Field(ge=0, le=1, description="Probability of occurrence")
    mitigation_strategy: Optional[str] = Field(max_length=500)

class CompanyProfile(BaseModel):
    company_name: str = Field(max_length=200)
    ticker_symbol: str = Field(pattern=r"^[A-Z]{1,5}$", description="Stock ticker symbol")
    industry: IndustryClassification
    market_cap: float = Field(ge=0, description="Market capitalization in millions USD")
    employees: int = Field(ge=1, description="Number of employees")
    founded_year: int = Field(ge=1800, le=2024, description="Year company was founded")

class RiskAssessment(BaseModel):
    assessment_id: str = Field(pattern=r"^RA-\d{6}$", description="Risk assessment ID")
    company: CompanyProfile
    financial_metrics: FinancialMetrics
    credit_rating: CreditRating
    overall_risk_category: RiskCategory
    risk_factors: List[RiskFactor] = Field(min_items=3, max_items=10)
    assessment_date: str = Field(pattern=r"^\d{4}-\d{2}-\d{2}$")
    analyst_name: str = Field(max_length=100)
    confidence_level: float = Field(ge=0, le=1, description="Analyst confidence 0-1")
    
    @validator('financial_metrics')
    def validate_financial_consistency(cls, v):
        # Ensure debt_to_equity ratio makes sense
        if v.debt_to_equity > 5:
            raise ValueError('Debt to equity ratio seems too high')
        return v

# Display the complex schema
schema = RiskAssessment.model_json_schema()
print("Complex Risk Assessment Schema (abbreviated):")
print(json.dumps({k: v for k, v in schema.items() if k in ['title', 'type', 'properties']}, indent=2)[:1000] + "...")

## 2. Generate Complex Risk Assessment

Now let's generate a comprehensive risk assessment using our complex schema:

In [ ]:
# Create structured generator for risk assessment
risk_generator = outlines.generate.json(model, RiskAssessment)

# Complex prompt for risk assessment
risk_prompt = """
Generate a comprehensive risk assessment for TechCorp Inc., a large technology company 
with ticker symbol TECH. The company was founded in 1995, has 50,000 employees, and 
a market cap of around $80 billion. They have strong revenue of $25 billion with good 
EBITDA margins. The company has moderate debt levels and good liquidity. 
Consider technology sector risks, competitive pressures, and regulatory challenges.
Assessment should be performed by analyst John Smith on 2024-07-11.
"""

print("Generating complex risk assessment...")
start_time = time.time()
risk_assessment = risk_generator(risk_prompt)
generation_time = time.time() - start_time

print(f"\nGenerated in {generation_time:.2f} seconds")
print("\nComplex Risk Assessment:")
print("=" * 50)
print(json.dumps(risk_assessment, indent=2))

## 3. Validation and Analysis

Let's validate our complex output and analyze its components:

In [ ]:
# Validate and analyze the risk assessment
try:
    assessment = RiskAssessment(**risk_assessment)
    
    print("Validation Results:")
    print("✓ Schema validation: PASSED")
    print(f"✓ Assessment ID: {assessment.assessment_id}")
    print(f"✓ Company: {assessment.company.company_name} ({assessment.company.ticker_symbol})")
    print(f"✓ Industry: {assessment.company.industry.value}")
    print(f"✓ Credit Rating: {assessment.credit_rating.value}")
    print(f"✓ Overall Risk: {assessment.overall_risk_category.value}")
    print(f"✓ Number of risk factors: {len(assessment.risk_factors)}")
    print(f"✓ Analyst confidence: {assessment.confidence_level:.2%}")
    
    # Analyze risk factors
    print("\nRisk Factor Analysis:")
    total_risk_score = 0
    for i, factor in enumerate(assessment.risk_factors, 1):
        risk_value = factor.impact_score * factor.probability
        total_risk_score += risk_value
        print(f"{i}. {factor.factor_name}")
        print(f"   Impact: {factor.impact_score}/10, Probability: {factor.probability:.2%}")
        print(f"   Risk Value: {risk_value:.2f}")
    
    print(f"\nTotal Calculated Risk Score: {total_risk_score:.2f}")
    
    # Financial health indicators
    print("\nFinancial Health Indicators:")
    metrics = assessment.financial_metrics
    print(f"Revenue: ${metrics.revenue:,.0f}M")
    print(f"EBITDA: ${metrics.ebitda:,.0f}M")
    print(f"EBITDA Margin: {(metrics.ebitda/metrics.revenue)*100:.1f}%")
    print(f"Debt-to-Equity: {metrics.debt_to_equity:.2f}")
    print(f"Current Ratio: {metrics.current_ratio:.2f}")
    print(f"ROE: {metrics.return_on_equity:.1f}%")
    
except Exception as e:
    print(f"Validation failed: {e}")

## 4. Regular Expression Constraints

Let's explore using regular expressions for more specific formatting constraints:

In [ ]:
# Define a model with regex constraints for financial identifiers
class FinancialIdentifiers(BaseModel):
    isin: str = Field(pattern=r"^[A-Z]{2}[A-Z0-9]{9}[0-9]$", description="ISIN code")
    cusip: str = Field(pattern=r"^[0-9]{3}[0-9A-Z]{5}[0-9]$", description="CUSIP code")
    lei: str = Field(pattern=r"^[A-Z0-9]{18}[0-9]{2}$", description="Legal Entity Identifier")
    swift_bic: str = Field(pattern=r"^[A-Z]{4}[A-Z]{2}[A-Z0-9]{2}([A-Z0-9]{3})?$", description="SWIFT BIC code")
    iban: str = Field(pattern=r"^[A-Z]{2}[0-9]{2}[A-Z0-9]{4}[0-9]{7}([A-Z0-9]?){0,16}$", description="IBAN")

# Create generator for financial identifiers
id_generator = outlines.generate.json(model, FinancialIdentifiers)

id_prompt = """
Generate valid financial identifiers for a US technology company listed on NASDAQ.
Include proper ISIN, CUSIP, LEI, SWIFT BIC, and IBAN codes following international standards.
"""

print("Generating financial identifiers with regex constraints...")
identifiers = id_generator(id_prompt)

print("\nGenerated Financial Identifiers:")
print(json.dumps(identifiers, indent=2))

## 5. Performance Comparison

Let's compare generation performance across different complexity levels:

In [ ]:
# Define schemas of varying complexity
class SimpleSchema(BaseModel):
    name: str
    value: float

class MediumSchema(BaseModel):
    id: str = Field(pattern=r"^[A-Z]{3}-\d{3}$")
    category: Literal["A", "B", "C"]
    metrics: List[float] = Field(min_items=2, max_items=5)

# Performance test function
def performance_test(schema_class, prompt, iterations=3):
    generator = outlines.generate.json(model, schema_class)
    times = []
    
    for i in range(iterations):
        start_time = time.time()
        result = generator(prompt)
        end_time = time.time()
        times.append(end_time - start_time)
    
    return {
        "avg_time": sum(times) / len(times),
        "min_time": min(times),
        "max_time": max(times),
        "sample_output": result
    }

# Run performance tests
print("Performance Comparison:")
print("=" * 50)

# Simple schema test
simple_result = performance_test(SimpleSchema, "Generate a simple financial metric", 2)
print(f"Simple Schema - Avg: {simple_result['avg_time']:.2f}s")

# Medium schema test  
medium_result = performance_test(MediumSchema, "Generate medium complexity data", 2)
print(f"Medium Schema - Avg: {medium_result['avg_time']:.2f}s")

# Complex schema (using our risk assessment)
complex_prompt = "Generate a brief risk assessment for a small tech company"
complex_start = time.time()
complex_output = risk_generator(complex_prompt)
complex_time = time.time() - complex_start
print(f"Complex Schema - Time: {complex_time:.2f}s")

print(f"\nComplexity vs Performance:")
print(f"Simple → Medium: {medium_result['avg_time'] / simple_result['avg_time']:.1f}x slower")
print(f"Medium → Complex: {complex_time / medium_result['avg_time']:.1f}x slower")

## 6. Quality Assessment

Let's assess the quality and consistency of structured vs unstructured generation:

In [ ]:
# Generate multiple samples to test consistency
def consistency_test(generator, prompt, iterations=3):
    results = []
    for i in range(iterations):
        result = generator(prompt)
        results.append(result)
    return results

# Test with simple portfolio schema
class SimplePortfolio(BaseModel):
    client_id: str = Field(pattern=r"^[A-Z]{3}-\d{3}$")
    total_value: float = Field(ge=0)
    num_assets: int = Field(ge=1, le=20)

portfolio_generator = outlines.generate.json(model, SimplePortfolio)
portfolio_prompt = "Generate portfolio data for client ABC-123 with moderate holdings"

# Test consistency
print("Consistency Test - 3 Generations:")
print("=" * 40)

consistent_results = consistency_test(portfolio_generator, portfolio_prompt, 3)
for i, result in enumerate(consistent_results, 1):
    print(f"Generation {i}:")
    print(json.dumps(result, indent=2))
    print()

# Analyze consistency
client_ids = [r['client_id'] for r in consistent_results]
total_values = [r['total_value'] for r in consistent_results]
num_assets = [r['num_assets'] for r in consistent_results]

print("Consistency Analysis:")
print(f"Client IDs: {set(client_ids)} (unique: {len(set(client_ids))})")
print(f"Total values range: ${min(total_values):,.0f} - ${max(total_values):,.0f}")
print(f"Number of assets range: {min(num_assets)} - {max(num_assets)}")
print(f"All results valid: {all(r for r in consistent_results)}")

## 7. Real-World Financial Application

Let's create a practical example for ESG (Environmental, Social, Governance) reporting:

In [ ]:
# ESG Reporting Schema
class ESGMetric(BaseModel):
    category: Literal["environmental", "social", "governance"]
    metric_name: str = Field(max_length=100)
    value: float = Field(description="Metric value")
    unit: str = Field(max_length=50, description="Unit of measurement")
    target: Optional[float] = Field(description="Target value if applicable")
    performance: Literal["exceeds", "meets", "below", "not_applicable"]

class ESGReport(BaseModel):
    report_id: str = Field(pattern=r"^ESG-\d{4}-\d{6}$")
    company_name: str = Field(max_length=200)
    reporting_period: str = Field(pattern=r"^\d{4}$", description="Year")
    overall_esg_score: float = Field(ge=0, le=100, description="ESG score 0-100")
    environmental_score: float = Field(ge=0, le=100)
    social_score: float = Field(ge=0, le=100)
    governance_score: float = Field(ge=0, le=100)
    metrics: List[ESGMetric] = Field(min_items=5, max_items=15)
    certification: Optional[Literal["B_Corp", "LEED", "ISO_14001", "none"]]
    report_date: str = Field(pattern=r"^\d{4}-\d{2}-\d{2}$")

# Generate ESG report
esg_generator = outlines.generate.json(model, ESGReport)

esg_prompt = """
Generate an ESG sustainability report for GreenTech Solutions, a renewable energy company 
with strong environmental performance and good governance practices. They achieved B Corp 
certification and have been working on improving social metrics. Report for 2023, 
generated on 2024-07-11. Include metrics like carbon emissions, employee satisfaction, 
board diversity, energy efficiency, and community investment.
"""

print("Generating ESG Report...")
esg_report = esg_generator(esg_prompt)

print("\nESG Sustainability Report:")
print("=" * 50)
print(json.dumps(esg_report, indent=2))

## 8. Summary and Best Practices

Let's summarize what we've learned about advanced structured generation:

In [ ]:
# Create a summary of best practices
best_practices = {
    "Schema Design": [
        "Use specific field constraints (patterns, ranges, enums)",
        "Include descriptive field documentation", 
        "Leverage Pydantic validators for business logic",
        "Design nested structures for complex data",
        "Use enums for controlled vocabularies"
    ],
    "Performance Optimization": [
        "Start with simpler schemas and add complexity gradually",
        "Consider the trade-off between constraints and speed",
        "Use appropriate field limits (min/max items)",
        "Test with smaller models first",
        "Cache generators for repeated use"
    ],
    "Financial Applications": [
        "Ensure data consistency with validators",
        "Use proper financial identifier formats",
        "Include regulatory compliance constraints",
        "Design for integration with existing systems",
        "Consider audit trail requirements"
    ],
    "Quality Assurance": [
        "Test generation consistency across runs",
        "Validate outputs against business rules",
        "Monitor for edge cases and errors",
        "Implement fallback strategies",
        "Regular schema updates as requirements evolve"
    ]
}

print("Best Practices for Structured Generation:")
print("=" * 50)

for category, practices in best_practices.items():
    print(f"\n{category}:")
    for practice in practices:
        print(f"  • {practice}")

print("\n" + "=" * 50)
print("Key Takeaways:")
print("• Structured generation provides reliability and consistency")
print("• Complex schemas require more compute but offer better control")
print("• Regular expressions enable precise format control")
print("• Validation ensures business rule compliance")
print("• Performance scales with schema complexity")

## Exercise

Design and implement a schema for a **Regulatory Filing Report** that includes:

1. Company identification (multiple identifier types)
2. Filing type and period
3. Financial statements with validation
4. Risk disclosures
5. Management certifications

Generate a sample report and validate it meets regulatory requirements.